# zarr-java Core Library — Integration Tests

Verifies that the code examples from `USERGUIDE.md` work against the locally-built JARs.

**Start via** `./notebooks/start.sh` from the repo root — that builds the project, wires up the classpath, and opens JupyterLab. No internet access needed for the library.

Run cells top-to-bottom with **Run All Cells** or `Shift+Enter`.

## Setup

In [1]:
String REPO_ROOT = System.getenv("REPO_ROOT");
if (REPO_ROOT == null) throw new IllegalStateException(
    "REPO_ROOT not set — start JupyterLab via ./notebooks/start.sh");
System.out.println("Repo root : " + REPO_ROOT);
System.out.println("Library   : zarr-java-core (loaded via IJAVA_CLASSPATH)");

Repo root : /home/konstantin/Documents/zarr-java
Library   : zarr-java-core (loaded via IJAVA_CLASSPATH)


## 1 · v3 Array — MemoryStore: create, write, read

In [11]:
import dev.zarr.zarrjava.v3.Array;
import dev.zarr.zarrjava.v3.DataType;
import dev.zarr.zarrjava.store.MemoryStore;
import dev.zarr.zarrjava.store.StoreHandle;

StoreHandle memHandle = new MemoryStore().resolve("myarray");
Array array = Array.create(
    memHandle,
    Array.metadataBuilder()
        .withShape(100, 100)
        .withDataType(DataType.FLOAT32)
        .withChunkShape(10, 10)
        .withFillValue(0.0f)
        .build()
);

ucar.ma2.Array data = ucar.ma2.Array.factory(
    ucar.ma2.DataType.FLOAT, new int[]{100, 100});
java.util.Arrays.fill((float[]) data.getStorage(), 1.0f);
array.write(data);

ucar.ma2.Array readBack = array.read();
System.out.println("Shape     : " + java.util.Arrays.toString(readBack.getShape()));
System.out.println("Element[0]: " + readBack.getFloat(0));

Shape     : [100, 100]
Element[0]: 1.0


## 2 · v3 Array — FilesystemStore: create, write, read subset

In [12]:
import dev.zarr.zarrjava.store.FilesystemStore;

new ProcessBuilder("rm", "-rf", "/tmp/zarr_core_fs").start().waitFor();
StoreHandle fsHandle = new FilesystemStore("/tmp/zarr_core_fs").resolve("myarray");

Array fsArray = Array.create(
    fsHandle,
    Array.metadataBuilder()
        .withShape(50, 50, 50)
        .withDataType(DataType.INT32)
        .withChunkShape(10, 10, 10)
        .withFillValue(0)
        .build()
);

ucar.ma2.Array writeData = ucar.ma2.Array.factory(
    ucar.ma2.DataType.INT, new int[]{10, 10, 10});
java.util.Arrays.fill((int[]) writeData.getStorage(), 42);
fsArray.write(new long[]{0, 0, 0}, writeData);

ucar.ma2.Array subset = fsArray.read(new long[]{0, 0, 0}, new long[]{10, 10, 10});
System.out.println("Written value : 42");
System.out.println("Read back [0] : " + subset.getInt(0));
System.out.println("Subset shape  : " + java.util.Arrays.toString(subset.getShape()));

Written value : 42
Read back [0] : 42
Subset shape  : [10, 10, 10]


## 3 · v3 Codecs — Blosc, Gzip, Zstd

In [13]:
MemoryStore codecStore = new MemoryStore();

Array bloscArray = Array.create(
    codecStore.resolve("blosc"),
    Array.metadataBuilder()
        .withShape(100, 100).withDataType(DataType.FLOAT32)
        .withChunkShape(50, 50).withFillValue(0.0f)
        .withCodecs(c -> c.withBlosc("zstd", 5))
        .build()
);
bloscArray.write(data);
System.out.println("Blosc  read[0]: " + bloscArray.read().getFloat(0));

Array gzipArray = Array.create(
    codecStore.resolve("gzip"),
    Array.metadataBuilder()
        .withShape(100, 100).withDataType(DataType.FLOAT32)
        .withChunkShape(50, 50).withFillValue(0.0f)
        .withCodecs(c -> c.withGzip(6))
        .build()
);
gzipArray.write(data);
System.out.println("Gzip   read[0]: " + gzipArray.read().getFloat(0));

Array zstdArray = Array.create(
    codecStore.resolve("zstd"),
    Array.metadataBuilder()
        .withShape(100, 100).withDataType(DataType.FLOAT32)
        .withChunkShape(50, 50).withFillValue(0.0f)
        .withCodecs(c -> c.withZstd(3))
        .build()
);
zstdArray.write(data);
System.out.println("Zstd   read[0]: " + zstdArray.read().getFloat(0));

Blosc  read[0]: 1.0
Gzip   read[0]: 1.0
Zstd   read[0]: 1.0


## 4 · v3 Sharding

In [14]:
Array shardedArray = Array.create(
    new MemoryStore().resolve("sharded"),
    Array.metadataBuilder()
        .withShape(100, 100, 100)
        .withDataType(DataType.UINT8)
        .withChunkShape(20, 20, 20)
        .withFillValue((byte) 0)
        .withCodecs(c -> c.withSharding(
            new int[]{10, 10, 10},
            inner -> inner.withBlosc("zstd", 5)
        ))
        .build()
);

ucar.ma2.Array chunkData = ucar.ma2.Array.factory(
    ucar.ma2.DataType.UBYTE, new int[]{20, 20, 20});
java.util.Arrays.fill((byte[]) chunkData.getStorage(), (byte) 7);
shardedArray.write(new long[]{0, 0, 0}, chunkData);

ucar.ma2.Array shardRead = shardedArray.read(new long[]{0,0,0}, new long[]{20,20,20});
System.out.println("Sharded shape : " + java.util.Arrays.toString(shardRead.getShape()));
System.out.println("Sharded val[0]: " + shardRead.getByte(0));

Sharded shape : [20, 20, 20]
Sharded val[0]: 7


## 5 · v3 Groups — create hierarchy, navigate, access members

In [15]:
import dev.zarr.zarrjava.v3.Group;

new ProcessBuilder("rm", "-rf", "/tmp/zarr_groups").start().waitFor();
StoreHandle groupRoot = new FilesystemStore("/tmp/zarr_groups").resolve();
Group root = Group.create(groupRoot);

Group dataGroup = root.createGroup("data");
Group metaGroup = root.createGroup("metadata");

Array childArray = dataGroup.createArray(
    "raw",
    Array.metadataBuilder()
        .withShape(1000, 1000)
        .withDataType(DataType.UINT16)
        .withChunkShape(100, 100)
        .withFillValue(0)
        .build()
);

long rootCount = root.list().count();
long dataCount = dataGroup.list().count();
Array retrieved = (Array) dataGroup.get("raw");

System.out.println("Root member count : " + rootCount);
System.out.println("data/ member count: " + dataCount);
System.out.println("Child array shape : " + java.util.Arrays.toString(retrieved.metadata().shape));
System.out.println("Child array dtype : " + retrieved.metadata().dataType);

Root member count : 3
data/ member count: 1
Child array shape : [1000, 1000]
Child array dtype : UINT16


## 6 · Attributes — set and read

In [16]:
import dev.zarr.zarrjava.core.Attributes;

Array attrArray = Array.create(
    new MemoryStore().resolve("withattrs"),
    Array.metadataBuilder()
        .withShape(10, 10)
        .withDataType(DataType.INT32)
        .withChunkShape(5, 5)
        .withFillValue(0)
        .build()
);

Attributes attrs = new Attributes();
attrs.put("description", "my dataset");
attrs.put("units", "meters");
attrs.put("version", 2);
attrArray = attrArray.setAttributes(attrs);

Attributes readAttrs = attrArray.metadata().attributes();
System.out.println("description: " + readAttrs.get("description"));
System.out.println("units      : " + readAttrs.get("units"));
System.out.println("version    : " + readAttrs.get("version"));

description: my dataset
units      : meters
version    : 2


## 7 · v2 Array — create, write, read with Blosc and Zlib

In [17]:
// Use fully-qualified names to avoid conflict with v3 Array/DataType imported above
dev.zarr.zarrjava.v2.Array v2Blosc = dev.zarr.zarrjava.v2.Array.create(
    new MemoryStore().resolve("v2blosc"),
    dev.zarr.zarrjava.v2.Array.metadataBuilder()
        .withShape(10, 10)
        .withDataType(dev.zarr.zarrjava.v2.DataType.UINT8)
        .withChunks(5, 5)
        .withFillValue(1)
        .withBloscCompressor("lz4", "shuffle", 5)
        .build()
);
ucar.ma2.Array v2Data = ucar.ma2.Array.factory(
    ucar.ma2.DataType.UBYTE, new int[]{10, 10});
java.util.Arrays.fill((byte[]) v2Data.getStorage(), (byte) 9);
v2Blosc.write(v2Data);
System.out.println("v2 Blosc shape  : " + java.util.Arrays.toString(v2Blosc.read().getShape()));
System.out.println("v2 Blosc val[0] : " + v2Blosc.read().getByte(0));

dev.zarr.zarrjava.v2.Array v2Zlib = dev.zarr.zarrjava.v2.Array.create(
    new MemoryStore().resolve("v2zlib"),
    dev.zarr.zarrjava.v2.Array.metadataBuilder()
        .withShape(15, 10)
        .withDataType(dev.zarr.zarrjava.v2.DataType.UINT8)
        .withChunks(5, 5)
        .withFillValue(0)
        .withZlibCompressor(6)
        .build()
);
v2Zlib.write(ucar.ma2.Array.factory(ucar.ma2.DataType.UBYTE, new int[]{15, 10}));
System.out.println("v2 Zlib shape   : " + java.util.Arrays.toString(v2Zlib.read().getShape()));

v2 Blosc shape  : [10, 10]
v2 Blosc val[0] : 9
v2 Zlib shape   : [15, 10]


## 8 · ZIP Store — write with BufferedZipStore, read with ReadOnlyZipStore

In [18]:
import dev.zarr.zarrjava.store.BufferedZipStore;
import dev.zarr.zarrjava.store.ReadOnlyZipStore;

java.nio.file.Path zipPath = java.nio.file.Paths.get("/tmp/zarr_test.zip");
zipPath.toFile().delete();

try (BufferedZipStore zipWrite = new BufferedZipStore(zipPath)) {
    Array zipArray = Array.create(
        zipWrite.resolve("myarray"),
        Array.metadataBuilder()
            .withShape(100, 100)
            .withDataType(DataType.FLOAT32)
            .withChunkShape(10, 10)
            .withFillValue(0.0f)
            .build()
    );
    ucar.ma2.Array zipData = ucar.ma2.Array.factory(
        ucar.ma2.DataType.FLOAT, new int[]{100, 100});
    java.util.Arrays.fill((float[]) zipData.getStorage(), 3.14f);
    zipArray.write(zipData);
}  // auto-closes and flushes to disk

ReadOnlyZipStore zipRead = new ReadOnlyZipStore(zipPath);
Array zipReadArray = Array.open(zipRead.resolve("myarray"));
ucar.ma2.Array zipResult = zipReadArray.read();
System.out.println("ZIP shape  : " + java.util.Arrays.toString(zipResult.getShape()));
System.out.printf ("ZIP val[0] : %.2f%n", zipResult.getFloat(0));

ZIP shape  : [100, 100]
ZIP val[0] : 3.14


java.io.PrintStream@69564c9a

## 9 · Open existing testdata (v3 from repo)

In [19]:
Array existing = Array.open(
    new FilesystemStore(REPO_ROOT +
        "/zarr-java-core/testdata/sharding_index_location/end").resolve()
);
System.out.println("Testdata shape  : " + java.util.Arrays.toString(existing.metadata().shape));
System.out.println("Testdata dtype  : " + existing.metadata().dataType);
System.out.println("Testdata chunks : " + java.util.Arrays.toString(existing.metadata().chunkShape()));

Testdata shape  : [16, 16, 16]
Testdata dtype  : INT32
Testdata chunks : [16, 8, 8]
